# 08 · Flow matching condicional

Entrena un campo de velocidades condicionado al régimen con el objetivo de conditional flow matching, y muestrea integrando la ODE de transporte.

**Responsable:** Oscar

**Entradas**

- `data/processed/ventanas.npz`

**Salidas**

- `models/generadores/flow_matching/ (modelo.pkl o .keras, historial.csv, meta.json)`
- `data/synthetic/flow_matching.npz`

**Tiempo estimado:** ~30 min en CPU (250 épocas; el muestreo por ODE añade 50 pasos por lote).

**Independencia.** Este notebook solo lee `data/processed/ventanas.npz` (notebook 02) y solo escribe en `models/generadores/flow_matching/` y `data/synthetic/flow_matching.npz`. No depende de ningún otro notebook de generador ni de sus salidas, de modo que los notebooks 04 a 10 pueden ejecutarse en paralelo y en cualquier orden por distintas personas.

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import ventanas

part = ventanas.cargar_procesado()
train, val, test = part.train, part.val, part.test
print(train, val, test, sep="\n")

In [ ]:
from src import regimenes
from src.generadores import base

v = config.ventanas()
n_regimenes = config.n_regimenes()

bloque_train = ventanas.empaquetar(train)
print("bloque de train:", bloque_train.shape, "· d esperada:", ventanas.dimension_bloque(v))
regimenes.distribucion(train.y_reg, n_regimenes)

## Por qué flow matching

Transporta ruido gaussiano a datos siguiendo una trayectoria, igual que la difusión,
pero aprende el **campo de velocidades** de un camino recto en vez de una cadena de
denoising. El objetivo es una regresión, no una verosimilitud variacional ni un
juego: es estable, no tiene colapso de modo y no necesita las 1.000 evaluaciones de
red que exige un DDPM.

El régimen entra como condición del campo, no como algo a generar. Eso es lo que
permite fijar a voluntad la mezcla de clases del conjunto sintético.

Decisiones que importan en este dataset:

**Balanceo de clases.** Cada época se remuestrea con probabilidad inversa a la
frecuencia de la clase. Sin ello la rama de crisis recibe diez veces menos gradiente
y su campo queda mal estimado. Cambia la marginal implícita que aprende el modelo,
pero como después se genera *condicionando* en la clase, esa marginal es
irrelevante: el balanceo es beneficio neto.

**Validación interna sobre la cola cronológica.** Se reserva el último tramo del
bloque, no una muestra aleatoria, y se descartan 80 ventanas en el corte porque las
ventanas consecutivas se solapan. Es un indicador de convergencia, no una evaluación
seria; esa la da el split real del proyecto.

**EMA de los pesos.** El promedio móvil exponencial reduce la varianza residual del
último tramo del entrenamiento y da mejores muestras sin coste de inferencia.

El bucle de entrenamiento es un bucle de PyTorch explícito y no `model.fit`, porque
el objetivo de CFM se resortea en cada paso: `t`, `x_0` y por tanto `x_t` y `u` son
distintos cada vez que se ve la misma ventana.

In [ ]:
generador = base.instanciar(
    "flow_matching",
    n_regimenes=n_regimenes,
    ancho=512,
    n_capas=3,
    epocas=250,
    tam_lote=256,
    tasa_aprendizaje=5e-4,
    ema=0.999,
    balanceo_clases=True,
    n_pasos=50,
    metodo_integracion="euler",
    hilos_torch=4,
    verboso=True,
)
generador.fit(bloque_train, train.y_reg)
generador

## Convergencia

Aquí la pérdida sí es una regresión y sí debe bajar y aplanarse. La curva de
validación es la informativa: si se separa de la de entrenamiento, el campo se está
ajustando a las ventanas concretas de train en lugar de a la geometría del
transporte.

La pérdida no tiende a cero por construcción: el objetivo de CFM tiene varianza
irreducible, porque el mismo `x_t` puede provenir de pares `(x_0, x_1)` distintos y
la red solo puede predecir la velocidad media. Lo que importa es la meseta, no el
valor absoluto.

In [ ]:
fig, eje = plt.subplots()
viz.curva_convergencia(generador.historial, "Convergencia · " + generador.etiqueta, eje=eje)
viz.guardar(fig, "convergencia_flow_matching")

generador.historial.tail(3).round(4)

## Inspección visual

Proyección PCA de reales y sintéticos, con la PCA ajustada **solo con los reales**
para que los ejes describan la estructura del mercado y no la del generador.

Es la comprobación más rápida y la que detecta los dos fallos gruesos: si la nube
sintética no cubre la real, el generador ha colapsado a un modo; si la desborda
ampliamente, está inventando configuraciones de mercado que nunca ocurrieron.

Se mira el régimen de crisis porque es el que tiene menos datos reales y, por
tanto, donde el generador tiene más margen para desviarse.

In [ ]:
CRISIS = n_regimenes - 1

muestra_crisis = generador.generate(600, regimen=CRISIS)
reales_crisis = bloque_train[train.y_reg == CRISIS]

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
viz.real_vs_sintetico(bloque_train, generador.generate(600, regimen=0),
                      "{} · régimen de calma".format(generador.etiqueta), eje=ejes[0])
viz.real_vs_sintetico(reales_crisis, muestra_crisis,
                      "{} · régimen de crisis".format(generador.etiqueta), eje=ejes[1])
fig.tight_layout()
viz.guardar(fig, "pca_" + generador.nombre)

print("reales de crisis:", len(reales_crisis), "· sintéticos generados:", len(muestra_crisis))

## Banco de muestras

Se genera un banco uniforme por régimen y se exporta a `data/synthetic/`. La mezcla
concreta de cada dataset la decide el notebook 11 muestreando de este banco, no
volviendo a invocar al generador: así el barrido no necesita tener los siete
modelos cargados en memoria y dos ejecuciones del notebook 12 usan exactamente las
mismas muestras sintéticas.

El banco es uniforme —no replica el desbalance real— porque la política de reparto
es un grado de libertad del experimento y se aplica después.

In [ ]:
MUESTRAS_POR_REGIMEN = 2000

reparto = {k: MUESTRAS_POR_REGIMEN for k in range(n_regimenes)}
bloques_sint, y_sint = generador.generate_dataset(reparto)

print("banco:", bloques_sint.shape, "· etiquetas:", np.bincount(y_sint, minlength=n_regimenes))
print("rango de valores:", round(float(bloques_sint.min()), 2), "→",
      round(float(bloques_sint.max()), 2),
      "(referencia real:", round(float(bloque_train.min()), 2), "→",
      round(float(bloque_train.max()), 2), ")")

## Persistencia

`guardar()` deja el modelo, la curva de convergencia y los metadatos en
`models/generadores/`. Es lo que permite que el resto del grupo salte directamente
al análisis sin reentrenar nada.

In [ ]:
ruta_muestras = generador.exportar_muestras(bloques_sint, y_sint)
ruta_modelo = generador.guardar()

print("muestras:", ruta_muestras)
print("modelo:  ", ruta_modelo)
pd.Series(generador.resumen_convergencia()).round(4)

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_MODELOS_GEN / "flow_matching" / "meta.json",
    src.DIR_MODELOS_GEN / "flow_matching" / "historial.csv",
    src.DIR_SINTETICO / "flow_matching.npz",
    src.DIR_FIGURAS / "convergencia_flow_matching.png",
    src.DIR_FIGURAS / "pca_flow_matching.png",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
